# EXP 01 — Submission-Safe Rebuild of V5

Notebook ini membangun ulang pipeline prediksi football goals menjadi **100% submission-safe**. Fokus utamanya adalah:
- menghapus seluruh ketergantungan pada ground truth test,
- membangun ulang history features secara **anti-leak**,
- melakukan validasi **time-based, match-level, dan jujur**,
- mempertahankan struktur model kuat ala V5:
  - per-gender modeling,
  - LightGBM,
  - XGBoost,
  - CatBoost,
  - Ridge stacker,
  - MBR decoding,
  - per-gender Dixon-Coles rho.

Prinsip utamanya sederhana:  
**semua fitur train hanya boleh melihat masa lalu**, dan **semua fitur test hanya boleh berasal dari state terakhir train**.

> Catatan: notebook ini sengaja memisahkan jelas antara:
> 1. pembuatan fitur train anti-leak,  
> 2. pembuatan fitur test freeze-safe,  
> 3. evaluasi offline yang jujur di validation folds,  
> 4. generation submission final.


## 1. Setup and imports

Bagian ini menyiapkan environment, seed, dan konfigurasi eksperimen.  
Default di bawah dibuat agar notebook tetap realistis untuk dijalankan end-to-end. Kalau ingin eksperimen lebih berat, beberapa parameter bisa dinaikkan nanti.


In [1]:
import os
import gc
import json
import copy
import math
import random
import warnings
from collections import defaultdict, deque

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

CONFIG = {
    "seed": SEED,
    "n_splits": 3,
    "valid_frac": 0.10,
    "min_train_frac": 0.50,
    "fast_mode": True,
    "tune_on_first_fold_only": True,
    "run_optional_recursive_branch": False,
    "max_decode_goals": 8,
}

print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  - {k}: {v}")


CONFIG:
  - seed: 42
  - n_splits: 3
  - valid_frac: 0.1
  - min_train_frac: 0.5
  - fast_mode: True
  - tune_on_first_fold_only: True
  - run_optional_recursive_branch: False
  - max_decode_goals: 8


## 2. Load data

Notebook akan mencoba membaca file dari beberapa lokasi umum:
- `./data/...`
- current working directory
- `/mnt/data/...` (berguna saat notebook diuji di environment lain)

Ini dibuat defensif supaya notebook lebih mudah dipindah.


In [2]:
from pathlib import Path
import os

print("Current working dir:", os.getcwd())
print("Isi folder saat ini:")
for p in Path(".").iterdir():
    print("-", p)

Current working dir: /home/pityudhistira28/lomba/Hubert/Notebook
Isi folder saat ini:
- exp01_submission_safe_rebuild_v5.ipynb
- football_goal_prediction_v5.ipynb
- .ipynb_checkpoints


In [3]:
train = pd.read_csv('../dataset/train.csv')
test  = pd.read_csv('../dataset/test.csv')
sample_submission = pd.read_csv('../dataset/sample submission.csv')

required_train_cols = {
    "Id", "match_id", "date", "gender", "team", "opponent", "is_home", "neutral",
    "tournament", "venue_country", "team_goals", "opp_goals"
}
required_test_cols = {
    "Id", "match_id", "date", "gender", "team", "opponent", "is_home", "neutral",
    "tournament", "venue_country"
}

missing_train = required_train_cols - set(train.columns)
missing_test = required_test_cols - set(test.columns)

assert not missing_train, f"Kolom train wajib ada, tetapi missing: {missing_train}"
assert not missing_test, f"Kolom test wajib ada, tetapi missing: {missing_test}"
assert {"Id", "team_goals", "opp_goals"}.issubset(sample_submission.columns), "Format sample submission tidak valid."

train = train.sort_values(["date", "match_id", "Id"]).reset_index(drop=True)
test = test.sort_values(["date", "match_id", "Id"]).reset_index(drop=True)

train["row_idx_global"] = np.arange(len(train))
test["row_idx_global"] = np.arange(len(test))

print("train shape:", train.shape)
print("test shape :", test.shape)
print("sample shape:", sample_submission.shape)


train shape: (78772, 48)
test shape : (42422, 21)
sample shape: (42422, 3)


## 3. Short data audit

Audit di sini sengaja singkat, tapi tetap cukup untuk memastikan:
- train dan test terbaca dengan benar,
- struktur 2 row per match memang valid,
- distribusi gender dan tournament masuk akal,
- missing values penting terlihat dari awal.


In [4]:
def short_audit(train_df, test_df):
    print("=== BASIC SHAPE ===")
    print("train:", train_df.shape)
    print("test :", test_df.shape)
    print()

    print("=== DATE RANGE ===")
    print("train:", train_df["date"].min(), "->", train_df["date"].max())
    print("test :", test_df["date"].min(), "->", test_df["date"].max())
    print()

    print("=== GENDER DISTRIBUTION ===")
    print("train:")
    print(train_df["gender"].value_counts(dropna=False))
    print()
    print("test:")
    print(test_df["gender"].value_counts(dropna=False))
    print()

    print("=== TOP TOURNAMENTS (TRAIN) ===")
    print(train_df["tournament"].value_counts(dropna=False).head(15))
    print()

    print("=== STRUCTURE CHECK: rows per match_id ===")
    train_match_sizes = train_df.groupby("match_id").size().value_counts().sort_index()
    test_match_sizes = test_df.groupby("match_id").size().value_counts().sort_index()
    print("train rows-per-match distribution:")
    print(train_match_sizes)
    print("test rows-per-match distribution:")
    print(test_match_sizes)
    print()

    assert set(train_df.groupby("match_id").size().unique()) == {2}, "Train bukan 2 baris per pertandingan."
    assert set(test_df.groupby("match_id").size().unique()) == {2}, "Test bukan 2 baris per pertandingan."

    important_missing_cols = [
        c for c in [
            "altitude_venue", "temperature_venue", "population_team", "population_opp",
            "gdp_per_capita_team", "gdp_per_capita_opp", "rank_team", "rank_opponent"
        ] if c in train_df.columns or c in test_df.columns
    ]

    missing_rows = []
    for c in important_missing_cols:
        if c in train_df.columns:
            missing_rows.append(("train", c, float(train_df[c].isna().mean())))
        if c in test_df.columns:
            missing_rows.append(("test", c, float(test_df[c].isna().mean())))

    missing_df = pd.DataFrame(missing_rows, columns=["split", "column", "missing_ratio"])
    print("=== IMPORTANT MISSING RATIO ===")
    display(missing_df.sort_values(["column", "split"]).reset_index(drop=True))

    unique_teams = pd.concat([train_df["team"], train_df["opponent"]]).nunique()
    unique_teams_test = pd.concat([test_df["team"], test_df["opponent"]]).nunique()
    print("unique teams in train universe:", unique_teams)
    print("unique teams in test universe :", unique_teams_test)

short_audit(train, test)


=== BASIC SHAPE ===
train: (78772, 48)
test : (42422, 21)

=== DATE RANGE ===
train: 1872-11-30 -> 2011-08-04
test : 2011-08-06 -> 2026-03-31

=== GENDER DISTRIBUTION ===
train:
gender
M    69966
W     8806
Name: count, dtype: int64

test:
gender
M    28464
W    13958
Name: count, dtype: int64

=== TOP TOURNAMENTS (TRAIN) ===
tournament
Friendly                                28006
FIFA World Cup qualification            11934
UEFA Euro qualification                  5698
African Cup of Nations qualification     2734
FIFA World Cup                           1904
Copa América                             1660
AFC Asian Cup qualification              1264
Merdeka Tournament                       1190
British Home Championship                1046
African Cup of Nations                   1018
CFU Caribbean Cup qualification           992
CECAFA Cup                                992
Asian Games                               900
Island Games                              718
Algarve Cup      

,split,column,missing_ratio
0,test,altitude_venue,0.257084
1,train,altitude_venue,0.260930
2,test,gdp_per_capita_opp,0.345670
3,train,gdp_per_capita_opp,0.329521
4,test,gdp_per_capita_team,0.345670
5,train,gdp_per_capita_team,0.329521
6,test,population_opp,0.079487
7,train,population_opp,0.188798
8,test,population_team,0.079487
9,train,population_team,0.188798


unique teams in train universe: 294
unique teams in test universe : 315


## 4. AW-MAE metric

Evaluator offline di bawah dibuat **sejalan** dengan evaluator V5 yang kamu minta:
- base error = rata-rata absolute error 2 target,
- penalty untuk exact miss,
- penalty untuk outcome miss,
- penalty untuk goal-difference miss,
- multiplier tambahan jika outcome salah,
- nonlinear scaling,
- lalu dibobot dengan tournament weight.

Metrik ini dipakai di:
- tuning ringan,
- evaluasi fold,
- evaluasi akhir validation.


In [5]:
EXACT_PENALTY = 0.30
OUTCOME_PENALTY = 0.25
GD_PENALTY = 0.15
WRONG_OUTCOME_MULTIPLIER = 1.50
NONLINEAR_POWER = 1.50


def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()

    if "fifa world cup" in t or t == "world cup":
        return 2.00

    if (
        "afc championship" in t
        or "afc asian cup" in t
        or "asian cup" in t
        or "euro" in t
        or "copa" in t
        or "gold cup" in t
        or "afcon" in t
        or "nations" in t
        or "continental" in t
        or "championship" in t
    ):
        return 1.80

    if "friendly" in t:
        return 0.96

    return 1.20


def _outcome(a: int, b: int) -> int:
    if a > b:
        return 1
    if a < b:
        return -1
    return 0


def official_match_loss(
    y_team_true: int,
    y_opp_true: int,
    y_team_pred: int,
    y_opp_pred: int,
    tournament_weight: float = 1.0,
) -> float:
    base = (abs(int(y_team_true) - int(y_team_pred)) + abs(int(y_opp_true) - int(y_opp_pred))) / 2.0

    exact_hit = (int(y_team_true) == int(y_team_pred)) and (int(y_opp_true) == int(y_opp_pred))
    if not exact_hit:
        base += EXACT_PENALTY

    outcome_true = _outcome(int(y_team_true), int(y_opp_true))
    outcome_pred = _outcome(int(y_team_pred), int(y_opp_pred))
    if outcome_true != outcome_pred:
        base += OUTCOME_PENALTY

    gd_true = int(y_team_true) - int(y_opp_true)
    gd_pred = int(y_team_pred) - int(y_opp_pred)
    if gd_true != gd_pred:
        base += GD_PENALTY

    if outcome_true != outcome_pred:
        base *= WRONG_OUTCOME_MULTIPLIER

    return float(tournament_weight) * (base ** NONLINEAR_POWER)


def awmae_score(
    y_team_true,
    y_opp_true,
    y_team_pred,
    y_opp_pred,
    tournaments,
) -> float:
    losses = []
    for yt, yo, pt, po, t in zip(y_team_true, y_opp_true, y_team_pred, y_opp_pred, tournaments):
        w = get_tournament_weight(t)
        losses.append(official_match_loss(yt, yo, pt, po, w))
    return float(np.mean(losses))


def exact_hit_rate(y_team_true, y_opp_true, y_team_pred, y_opp_pred) -> float:
    return float(np.mean((np.asarray(y_team_true) == np.asarray(y_team_pred)) & (np.asarray(y_opp_true) == np.asarray(y_opp_pred))))


def outcome_hit_rate(y_team_true, y_opp_true, y_team_pred, y_opp_pred) -> float:
    true_outcomes = np.sign(np.asarray(y_team_true) - np.asarray(y_opp_true))
    pred_outcomes = np.sign(np.asarray(y_team_pred) - np.asarray(y_opp_pred))
    return float(np.mean(true_outcomes == pred_outcomes))


def gd_hit_rate(y_team_true, y_opp_true, y_team_pred, y_opp_pred) -> float:
    true_gd = np.asarray(y_team_true) - np.asarray(y_opp_true)
    pred_gd = np.asarray(y_team_pred) - np.asarray(y_opp_pred)
    return float(np.mean(true_gd == pred_gd))


print("Sanity check perfect prediction ->", awmae_score([1], [0], [1], [0], ["Friendly"]))
print("Sanity check same absolute error, world cup should be heavier than friendly:")
print("friendly:", awmae_score([1], [0], [0], [0], ["Friendly"]))
print("world cup:", awmae_score([1], [0], [0], [0], ["FIFA World Cup"]))


Sanity check perfect prediction -> 0.0
Sanity check same absolute error, world cup should be heavier than friendly:
friendly: 2.3183552790717816
world cup: 4.829906831399545


## 5. Submission-safe history state building

Ini inti notebook.

**Poin pentingnya:** data punya 2 baris per pertandingan.  
Kalau update state dilakukan setelah baris pertama, maka baris kedua akan bocor karena sudah melihat hasil match yang sama.

Karena itu, pipeline anti-leak di bawah bekerja **per match**, bukan per row:
1. ambil dua row dalam satu match,
2. bangun fitur keduanya dari state **sebelum** match,
3. baru setelah itu state di-update memakai hasil match tersebut.

Fitur yang dibangun:
- rolling form,
- H2H,
- Elo,
- Pi-rating,
- last-known rank proxy,
- days since last match,
- match counts.


In [6]:
SAFE_HISTORY_FEATURES = [
    "team_points_last5_safe",
    "opp_points_last5_safe",
    "points_last5_diff_safe",
    "team_points_last10_safe",
    "opp_points_last10_safe",
    "team_gd_last5_safe",
    "opp_gd_last5_safe",
    "gd_last5_diff_safe",
    "team_avg_goals_last5_safe",
    "team_avg_conceded_last5_safe",
    "opp_avg_goals_last5_safe",
    "opp_avg_conceded_last5_safe",
    "team_win_rate_last10_safe",
    "opp_win_rate_last10_safe",
    "days_since_last_match_team_safe",
    "days_since_last_match_opp_safe",
    "elo_team_safe",
    "elo_opp_safe",
    "elo_diff_safe",
    "pi_team_safe",
    "pi_opp_safe",
    "pi_diff_safe",
    "rank_team_last_known_safe",
    "rank_opp_last_known_safe",
    "rank_diff_last_known_safe",
    "rank_missing_team_last_known_safe",
    "rank_missing_opp_last_known_safe",
    "team_matches_played_safe",
    "opp_matches_played_safe",
    "h2h_points_last5_safe",
    "h2h_gd_last5_safe",
    "h2h_avg_goals_last5_safe",
    "h2h_avg_conceded_last5_safe",
    "h2h_matches_played_safe",
]


def points_for_score(goals_for: int, goals_against: int) -> int:
    if goals_for > goals_against:
        return 3
    if goals_for == goals_against:
        return 1
    return 0


def default_team_state():
    return {
        "points": deque(maxlen=10),
        "gd": deque(maxlen=10),
        "gf": deque(maxlen=10),
        "ga": deque(maxlen=10),
        "win": deque(maxlen=10),
        "last_date": None,
        "elo": 1500.0,
        "pi": 0.0,
        "last_rank": np.nan,
        "last_rank_missing": 1.0,
        "matches_played": 0.0,
    }


def default_h2h_state():
    return {
        "points": deque(maxlen=5),
        "gd": deque(maxlen=5),
        "gf": deque(maxlen=5),
        "ga": deque(maxlen=5),
        "matches_played": 0.0,
    }


def summarize_team_state(team_state, current_date):
    pts5 = sum(list(team_state["points"])[-5:]) if len(team_state["points"]) > 0 else np.nan
    pts10 = sum(team_state["points"]) if len(team_state["points"]) > 0 else np.nan
    gd5 = sum(list(team_state["gd"])[-5:]) if len(team_state["gd"]) > 0 else np.nan
    avg_gf5 = float(np.mean(list(team_state["gf"])[-5:])) if len(team_state["gf"]) > 0 else np.nan
    avg_ga5 = float(np.mean(list(team_state["ga"])[-5:])) if len(team_state["ga"]) > 0 else np.nan
    win_rate10 = float(np.mean(team_state["win"])) if len(team_state["win"]) > 0 else np.nan
    days_since = (
        (pd.Timestamp(current_date) - pd.Timestamp(team_state["last_date"])).days
        if team_state["last_date"] is not None
        else np.nan
    )
    return {
        "points_last5": pts5,
        "points_last10": pts10,
        "gd_last5": gd5,
        "avg_goals_last5": avg_gf5,
        "avg_conceded_last5": avg_ga5,
        "win_rate_last10": win_rate10,
        "days_since_last_match": days_since,
        "elo_pre": float(team_state["elo"]),
        "pi_pre": float(team_state["pi"]),
        "rank_last_known": team_state["last_rank"],
        "rank_last_known_missing": float(team_state["last_rank_missing"]),
        "matches_played": float(team_state["matches_played"]),
    }


def summarize_h2h_state(h2h_state):
    return {
        "h2h_points_last5_safe": sum(h2h_state["points"]) if len(h2h_state["points"]) > 0 else np.nan,
        "h2h_gd_last5_safe": sum(h2h_state["gd"]) if len(h2h_state["gd"]) > 0 else np.nan,
        "h2h_avg_goals_last5_safe": float(np.mean(h2h_state["gf"])) if len(h2h_state["gf"]) > 0 else np.nan,
        "h2h_avg_conceded_last5_safe": float(np.mean(h2h_state["ga"])) if len(h2h_state["ga"]) > 0 else np.nan,
        "h2h_matches_played_safe": float(h2h_state["matches_played"]),
    }


def elo_update(ra, rb, ga, gb, is_home=0, k=24.0, home_advantage=35.0):
    ra_eff = float(ra) + (home_advantage if int(is_home) == 1 else 0.0)
    expected_a = 1.0 / (1.0 + 10.0 ** ((float(rb) - ra_eff) / 400.0))
    score_a = 1.0 if ga > gb else 0.5 if ga == gb else 0.0
    margin_multiplier = 1.0 + 0.15 * min(abs(int(ga) - int(gb)), 6)
    delta = k * margin_multiplier * (score_a - expected_a)
    return float(ra) + delta, float(rb) - delta


def pi_update(pa, pb, ga, gb, lr=0.08, decay=0.92):
    goal_diff = int(ga) - int(gb)
    pa_new = decay * float(pa) + lr * goal_diff
    pb_new = decay * float(pb) - lr * goal_diff
    return pa_new, pb_new


def _allocate_history_feature_arrays(n_rows):
    return {name: np.full(n_rows, np.nan, dtype=float) for name in SAFE_HISTORY_FEATURES}


def _write_history_row(feature_arrays, idx, team_summary, opp_summary, h2h_summary):
    feature_arrays["team_points_last5_safe"][idx] = team_summary["points_last5"]
    feature_arrays["opp_points_last5_safe"][idx] = opp_summary["points_last5"]
    feature_arrays["points_last5_diff_safe"][idx] = (
        team_summary["points_last5"] - opp_summary["points_last5"]
        if pd.notna(team_summary["points_last5"]) and pd.notna(opp_summary["points_last5"])
        else np.nan
    )

    feature_arrays["team_points_last10_safe"][idx] = team_summary["points_last10"]
    feature_arrays["opp_points_last10_safe"][idx] = opp_summary["points_last10"]

    feature_arrays["team_gd_last5_safe"][idx] = team_summary["gd_last5"]
    feature_arrays["opp_gd_last5_safe"][idx] = opp_summary["gd_last5"]
    feature_arrays["gd_last5_diff_safe"][idx] = (
        team_summary["gd_last5"] - opp_summary["gd_last5"]
        if pd.notna(team_summary["gd_last5"]) and pd.notna(opp_summary["gd_last5"])
        else np.nan
    )

    feature_arrays["team_avg_goals_last5_safe"][idx] = team_summary["avg_goals_last5"]
    feature_arrays["team_avg_conceded_last5_safe"][idx] = team_summary["avg_conceded_last5"]
    feature_arrays["opp_avg_goals_last5_safe"][idx] = opp_summary["avg_goals_last5"]
    feature_arrays["opp_avg_conceded_last5_safe"][idx] = opp_summary["avg_conceded_last5"]

    feature_arrays["team_win_rate_last10_safe"][idx] = team_summary["win_rate_last10"]
    feature_arrays["opp_win_rate_last10_safe"][idx] = opp_summary["win_rate_last10"]

    feature_arrays["days_since_last_match_team_safe"][idx] = team_summary["days_since_last_match"]
    feature_arrays["days_since_last_match_opp_safe"][idx] = opp_summary["days_since_last_match"]

    feature_arrays["elo_team_safe"][idx] = team_summary["elo_pre"]
    feature_arrays["elo_opp_safe"][idx] = opp_summary["elo_pre"]
    feature_arrays["elo_diff_safe"][idx] = team_summary["elo_pre"] - opp_summary["elo_pre"]

    feature_arrays["pi_team_safe"][idx] = team_summary["pi_pre"]
    feature_arrays["pi_opp_safe"][idx] = opp_summary["pi_pre"]
    feature_arrays["pi_diff_safe"][idx] = team_summary["pi_pre"] - opp_summary["pi_pre"]

    feature_arrays["rank_team_last_known_safe"][idx] = team_summary["rank_last_known"]
    feature_arrays["rank_opp_last_known_safe"][idx] = opp_summary["rank_last_known"]
    feature_arrays["rank_diff_last_known_safe"][idx] = (
        team_summary["rank_last_known"] - opp_summary["rank_last_known"]
        if pd.notna(team_summary["rank_last_known"]) and pd.notna(opp_summary["rank_last_known"])
        else np.nan
    )
    feature_arrays["rank_missing_team_last_known_safe"][idx] = team_summary["rank_last_known_missing"]
    feature_arrays["rank_missing_opp_last_known_safe"][idx] = opp_summary["rank_last_known_missing"]

    feature_arrays["team_matches_played_safe"][idx] = team_summary["matches_played"]
    feature_arrays["opp_matches_played_safe"][idx] = opp_summary["matches_played"]

    for key, value in h2h_summary.items():
        feature_arrays[key][idx] = value


def update_states_from_observed_match(match_rows_df, team_states, h2h_states, use_rank_update=True):
    match_rows_df = match_rows_df.sort_values("Id").reset_index(drop=True)

    if len(match_rows_df) != 2:
        raise ValueError("Setiap match harus punya tepat 2 row saat update state.")

    row0 = match_rows_df.iloc[0]
    row1 = match_rows_df.iloc[1]

    g0f = int(row0["team_goals"])
    g0a = int(row0["opp_goals"])
    g1f = int(row1["team_goals"])
    g1a = int(row1["opp_goals"])

    assert g0f == g1a and g0a == g1f, "Dua row dalam match tidak saling mirror untuk skor."

    current_date = pd.Timestamp(row0["date"])

    for _, row in match_rows_df.iterrows():
        team = row["team"]
        opp = row["opponent"]
        goals_for = int(row["team_goals"])
        goals_against = int(row["opp_goals"])

        state = team_states[team]
        state["points"].append(points_for_score(goals_for, goals_against))
        state["gd"].append(goals_for - goals_against)
        state["gf"].append(goals_for)
        state["ga"].append(goals_against)
        state["win"].append(1 if goals_for > goals_against else 0)
        state["last_date"] = current_date
        state["matches_played"] += 1.0

        if use_rank_update and ("rank_team" in match_rows_df.columns):
            if pd.notna(row.get("rank_team", np.nan)):
                state["last_rank"] = float(row["rank_team"])
                state["last_rank_missing"] = 0.0
            else:
                state["last_rank_missing"] = 1.0 if pd.isna(state["last_rank"]) else 0.0

        h2h = h2h_states[(team, opp)]
        h2h["points"].append(points_for_score(goals_for, goals_against))
        h2h["gd"].append(goals_for - goals_against)
        h2h["gf"].append(goals_for)
        h2h["ga"].append(goals_against)
        h2h["matches_played"] += 1.0

    team_a = row0["team"]
    team_b = row1["team"]
    elo_a_new, elo_b_new = elo_update(
        team_states[team_a]["elo"],
        team_states[team_b]["elo"],
        g0f,
        g0a,
        is_home=int(row0["is_home"]),
    )
    team_states[team_a]["elo"] = elo_a_new
    team_states[team_b]["elo"] = elo_b_new

    pi_a_new, pi_b_new = pi_update(
        team_states[team_a]["pi"],
        team_states[team_b]["pi"],
        g0f,
        g0a,
    )
    team_states[team_a]["pi"] = pi_a_new
    team_states[team_b]["pi"] = pi_b_new


def build_train_history_states(train_df):
    train_df = train_df.sort_values(["date", "match_id", "Id"]).reset_index(drop=True).copy()

    team_states = defaultdict(default_team_state)
    h2h_states = defaultdict(default_h2h_state)

    feature_arrays = _allocate_history_feature_arrays(len(train_df))

    for _, match_rows in train_df.groupby("match_id", sort=False):
        match_rows = match_rows.sort_values("Id")
        current_date = pd.Timestamp(match_rows.iloc[0]["date"])

        for idx, row in match_rows.iterrows():
            team = row["team"]
            opp = row["opponent"]

            team_summary = summarize_team_state(team_states[team], current_date)
            opp_summary = summarize_team_state(team_states[opp], current_date)
            h2h_summary = summarize_h2h_state(h2h_states[(team, opp)])

            _write_history_row(feature_arrays, idx, team_summary, opp_summary, h2h_summary)

        update_states_from_observed_match(match_rows, team_states, h2h_states, use_rank_update=True)

    out = train_df.copy()
    for key, arr in feature_arrays.items():
        out[key] = arr

    return out, team_states, h2h_states


def make_test_features_freeze_safe(test_df, team_states, h2h_states):
    test_df = test_df.sort_values(["date", "match_id", "Id"]).reset_index(drop=True).copy()

    feature_arrays = _allocate_history_feature_arrays(len(test_df))

    for _, match_rows in test_df.groupby("match_id", sort=False):
        match_rows = match_rows.sort_values("Id")
        current_date = pd.Timestamp(match_rows.iloc[0]["date"])

        for idx, row in match_rows.iterrows():
            team = row["team"]
            opp = row["opponent"]

            team_summary = summarize_team_state(team_states[team], current_date)
            opp_summary = summarize_team_state(team_states[opp], current_date)
            h2h_summary = summarize_h2h_state(h2h_states[(team, opp)])

            _write_history_row(feature_arrays, idx, team_summary, opp_summary, h2h_summary)

    out = test_df.copy()
    for key, arr in feature_arrays.items():
        out[key] = arr

    return out


## 6. Feature engineering

Bagian ini membentuk fitur yang benar-benar aman dipakai saat submission.

Prinsipnya:
- **raw target dan identifier utama tidak dipakai sebagai feature**,
- `team` dan `opponent` sengaja **dibuang** dari feature space final agar notebook konsisten dengan constraint yang kamu minta,
- sinyal kekuatan tim tetap datang dari history states yang sudah dibangun secara anti-leak.


In [7]:
def clean_altitude(value):
    if pd.isna(value):
        return np.nan
    value = float(value)
    if value == -9999:
        return np.nan
    return value


def add_submission_safe_context_features(df):
    out = df.copy()

    if "altitude_venue" in out.columns:
        out["altitude_venue_clean"] = out["altitude_venue"].apply(clean_altitude)
    else:
        out["altitude_venue_clean"] = np.nan

    out["tournament_weight"] = out["tournament"].map(get_tournament_weight).astype(float)

    out["year"] = out["date"].dt.year.astype(int)
    out["month"] = out["date"].dt.month.astype(int)
    out["dayofyear"] = out["date"].dt.dayofyear.astype(int)
    out["dayofweek"] = out["date"].dt.dayofweek.astype(int)
    out["is_weekend"] = (out["dayofweek"] >= 5).astype(int)

    for col in [
        "population_team", "population_opp",
        "gdp_per_capita_team", "gdp_per_capita_opp",
        "distance_travel_team", "distance_travel_opp",
        "temperature_venue", "altitude_venue_clean",
    ]:
        if col not in out.columns:
            out[col] = np.nan

    out["population_team_log"] = np.log1p(out["population_team"].clip(lower=0))
    out["population_opp_log"] = np.log1p(out["population_opp"].clip(lower=0))
    out["population_log_diff"] = out["population_team_log"] - out["population_opp_log"]

    out["gdp_team_log"] = np.log1p(out["gdp_per_capita_team"].clip(lower=0))
    out["gdp_opp_log"] = np.log1p(out["gdp_per_capita_opp"].clip(lower=0))
    out["gdp_log_diff"] = out["gdp_team_log"] - out["gdp_opp_log"]

    out["travel_diff"] = out["distance_travel_team"] - out["distance_travel_opp"]
    out["travel_sum"] = out["distance_travel_team"] + out["distance_travel_opp"]

    out["recent_form_gap_safe"] = out["points_last5_diff_safe"]
    out["goal_form_gap_safe"] = out["team_avg_goals_last5_safe"] - out["opp_avg_goals_last5_safe"]
    out["defense_form_gap_safe"] = out["opp_avg_conceded_last5_safe"] - out["team_avg_conceded_last5_safe"]
    out["rest_gap_safe"] = out["days_since_last_match_opp_safe"] - out["days_since_last_match_team_safe"]

    out["elo_home_interaction_safe"] = out["elo_diff_safe"] * out["is_home"].astype(float)
    out["pi_home_interaction_safe"] = out["pi_diff_safe"] * out["is_home"].astype(float)

    out["rank_home_interaction_safe"] = out["rank_diff_last_known_safe"] * out["is_home"].astype(float)
    out["rank_missing_any_safe"] = (
        out["rank_missing_team_last_known_safe"].fillna(1.0)
        + out["rank_missing_opp_last_known_safe"].fillna(1.0)
    )

    out["confed_same_flag"] = (
        out["confederation_team"].astype(str).fillna("__MISSING__")
        == out["confederation_opp"].astype(str).fillna("__MISSING__")
    ).astype(int)

    out["altitude_x_temperature"] = out["altitude_venue_clean"] * out["temperature_venue"]
    out["altitude_x_neutral"] = out["altitude_venue_clean"] * out["neutral"].astype(float)

    out["history_strength_sum_safe"] = (
        out["team_points_last10_safe"].fillna(0)
        + out["opp_points_last10_safe"].fillna(0)
    )
    out["history_strength_gap_safe"] = (
        out["team_points_last10_safe"].fillna(0)
        - out["opp_points_last10_safe"].fillna(0)
    )

    return out


def make_train_features_submission_safe(train_df):
    train_hist, team_states, h2h_states = build_train_history_states(train_df)
    train_fe = add_submission_safe_context_features(train_hist)
    return train_fe, team_states, h2h_states


def make_test_features_freeze_safe_pipeline(test_df, team_states, h2h_states):
    test_hist = make_test_features_freeze_safe(test_df, team_states, h2h_states)
    test_fe = add_submission_safe_context_features(test_hist)
    return test_fe


def get_feature_space(train_fe, test_fe):
    hard_drop = {
        "Id", "match_id", "date", "row_idx_global",
        "team", "opponent", "gender",
        "team_goals", "opp_goals",
    }

    common_cols = sorted(set(train_fe.columns).intersection(test_fe.columns))
    feature_cols = [c for c in common_cols if c not in hard_drop]

    categorical_cols = [
        c for c in [
            "tournament",
            "venue_country",
            "confederation_team",
            "confederation_opp",
        ]
        if c in feature_cols
    ]
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    assert len(feature_cols) > 0, "Feature space kosong."
    return feature_cols, categorical_cols, numeric_cols


## 7. Build feature tables

Di sini kita benar-benar membangun:
- `train_fe`: fitur train anti-leak,
- `test_fe`: fitur test freeze-safe dari **last known state milik train saja**.

Jadi baseline utama notebook ini tetap sesuai prinsip submission-safe.


In [8]:
train["date"] = pd.to_datetime(train["date"], errors="coerce")
test["date"] = pd.to_datetime(test["date"], errors="coerce")

In [9]:
train_fe, train_team_states_final, train_h2h_states_final = make_train_features_submission_safe(train)
test_fe = make_test_features_freeze_safe_pipeline(
    test,
    team_states=copy.deepcopy(train_team_states_final),
    h2h_states=copy.deepcopy(train_h2h_states_final),
)

feature_cols, categorical_cols, numeric_cols = get_feature_space(train_fe, test_fe)

print("train_fe shape:", train_fe.shape)
print("test_fe shape :", test_fe.shape)
print("number of features:", len(feature_cols))
print("categorical features:", categorical_cols)
print("first 20 features:", feature_cols[:20])

assert set(feature_cols).issubset(train_fe.columns)
assert set(feature_cols).issubset(test_fe.columns)
assert sorted(feature_cols) == sorted([c for c in feature_cols if c in test_fe.columns]), "Train/test feature space tidak sinkron."


train_fe shape: (78772, 110)
test_fe shape : (42422, 83)
number of features: 76
categorical features: ['tournament', 'venue_country', 'confederation_team', 'confederation_opp']
first 20 features: ['altitude_venue', 'altitude_venue_clean', 'altitude_x_neutral', 'altitude_x_temperature', 'confed_same_flag', 'confederation_opp', 'confederation_team', 'dayofweek', 'dayofyear', 'days_since_last_match_opp_safe', 'days_since_last_match_team_safe', 'defense_form_gap_safe', 'distance_travel_opp', 'distance_travel_team', 'elo_diff_safe', 'elo_home_interaction_safe', 'elo_opp_safe', 'elo_team_safe', 'gd_last5_diff_safe', 'gdp_log_diff']


## 8. Temporal CV

Validasi dibangun pada level **match**, bukan row.  
Tujuannya:
- tidak ada match yang terbelah antara train dan valid,
- train fold selalu lebih awal dari validation fold,
- tidak ada leakage via match overlap.


In [10]:
def make_match_time_folds(df, n_splits=3, valid_frac=0.10, min_train_frac=0.50):
    df = df.sort_values(["date", "match_id", "Id"]).reset_index(drop=True).copy()

    match_calendar = (
        df.groupby("match_id", as_index=False)["date"]
        .min()
        .sort_values(["date", "match_id"])
        .reset_index(drop=True)
    )

    n_matches = len(match_calendar)
    valid_size = max(1, int(round(n_matches * valid_frac)))
    earliest_valid_start = max(int(round(n_matches * min_train_frac)), n_matches - n_splits * valid_size)

    folds = []
    for split_id in range(n_splits):
        valid_start = earliest_valid_start + split_id * valid_size
        valid_end = min(valid_start + valid_size, n_matches)

        if valid_start >= n_matches or valid_end <= valid_start:
            continue

        train_match_ids = set(match_calendar.loc[: valid_start - 1, "match_id"])
        valid_match_ids = set(match_calendar.loc[valid_start : valid_end - 1, "match_id"])

        if len(train_match_ids) == 0 or len(valid_match_ids) == 0:
            continue

        train_idx = df.index[df["match_id"].isin(train_match_ids)].to_numpy()
        valid_idx = df.index[df["match_id"].isin(valid_match_ids)].to_numpy()

        train_dates = df.loc[train_idx, "date"]
        valid_dates = df.loc[valid_idx, "date"]

        assert set(df.loc[train_idx, "match_id"]).isdisjoint(set(df.loc[valid_idx, "match_id"])), "Ada overlap match_id antar fold."
        assert train_dates.max() <= valid_dates.min(), "Train fold lebih baru daripada validation fold."

        folds.append(
            {
                "fold": len(folds),
                "train_idx": train_idx,
                "valid_idx": valid_idx,
                "train_start": train_dates.min(),
                "train_end": train_dates.max(),
                "valid_start": valid_dates.min(),
                "valid_end": valid_dates.max(),
                "n_train_rows": int(len(train_idx)),
                "n_valid_rows": int(len(valid_idx)),
                "n_train_matches": int(df.loc[train_idx, "match_id"].nunique()),
                "n_valid_matches": int(df.loc[valid_idx, "match_id"].nunique()),
            }
        )

    assert len(folds) > 0, "Fold temporal gagal dibangun."
    return folds


def fold_summary_df(folds):
    return pd.DataFrame(
        [
            {
                "fold": f["fold"],
                "train_rows": f["n_train_rows"],
                "valid_rows": f["n_valid_rows"],
                "train_matches": f["n_train_matches"],
                "valid_matches": f["n_valid_matches"],
                "train_start": f["train_start"],
                "train_end": f["train_end"],
                "valid_start": f["valid_start"],
                "valid_end": f["valid_end"],
            }
            for f in folds
        ]
    )


folds_all = make_match_time_folds(
    train_fe,
    n_splits=CONFIG["n_splits"],
    valid_frac=CONFIG["valid_frac"],
    min_train_frac=CONFIG["min_train_frac"],
)

display(fold_summary_df(folds_all))


,fold,train_rows,valid_rows,train_matches,valid_matches,train_start,train_end,valid_start,valid_end
0,0,55138,7878,27569,3939,1872-11-30,2001-07-23,2001-07-23,2005-01-30
1,1,63016,7878,31508,3939,1872-11-30,2005-01-30,2005-02-01,2008-06-06
2,2,70894,7878,35447,3939,1872-11-30,2008-06-06,2008-06-06,2011-08-04


## 9. Preprocessing, tuning, dan training helpers

Di sini ada helper untuk:
- preprocessing numerik + kategorikal,
- model builder,
- tuning ringan berbasis skor yang lebih dekat ke AW-MAE final,
- training CV per gender.

Tuning dibuat efisien, bukan brutal, karena fokus utama eksperimen ini adalah baseline submission-safe yang jujur.


In [11]:
def fit_preprocessor(train_part, feature_cols, categorical_cols):
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    num_imputer = SimpleImputer(strategy="median")
    X_num_train = num_imputer.fit_transform(train_part[numeric_cols])

    encoder = None
    if len(categorical_cols) > 0:
        train_cat = train_part[categorical_cols].copy().astype("object").fillna("__MISSING__")
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-1,
        )
        X_cat_train = encoder.fit_transform(train_cat)
    else:
        X_cat_train = np.empty((len(train_part), 0), dtype=float)

    X_train = np.hstack([X_num_train, X_cat_train]).astype(np.float32)

    bundle = {
        "feature_cols": feature_cols,
        "categorical_cols": categorical_cols,
        "numeric_cols": numeric_cols,
        "num_imputer": num_imputer,
        "cat_encoder": encoder,
        "model_feature_names": numeric_cols + categorical_cols,
    }
    return X_train, bundle


def transform_with_preprocessor(df_part, bundle):
    numeric_cols = bundle["numeric_cols"]
    categorical_cols = bundle["categorical_cols"]

    X_num = bundle["num_imputer"].transform(df_part[numeric_cols])

    if len(categorical_cols) > 0:
        cat_block = df_part[categorical_cols].copy().astype("object").fillna("__MISSING__")
        X_cat = bundle["cat_encoder"].transform(cat_block)
    else:
        X_cat = np.empty((len(df_part), 0), dtype=float)

    X = np.hstack([X_num, X_cat]).astype(np.float32)
    return X


def get_model_candidates(fast_mode=True):
    if fast_mode:
        return {
            "lgbm": [
                {
                    "n_estimators": 260,
                    "learning_rate": 0.04,
                    "num_leaves": 31,
                    "subsample": 0.90,
                    "colsample_bytree": 0.85,
                    "min_child_samples": 40,
                },
                {
                    "n_estimators": 340,
                    "learning_rate": 0.03,
                    "num_leaves": 63,
                    "subsample": 0.85,
                    "colsample_bytree": 0.90,
                    "min_child_samples": 30,
                },
            ],
            "xgb": [
                {
                    "n_estimators": 260,
                    "learning_rate": 0.04,
                    "max_depth": 6,
                    "min_child_weight": 4,
                    "subsample": 0.85,
                    "colsample_bytree": 0.85,
                    "reg_lambda": 1.0,
                },
                {
                    "n_estimators": 340,
                    "learning_rate": 0.03,
                    "max_depth": 5,
                    "min_child_weight": 3,
                    "subsample": 0.90,
                    "colsample_bytree": 0.90,
                    "reg_lambda": 1.5,
                },
            ],
            "cat": [
                {
                    "iterations": 260,
                    "learning_rate": 0.04,
                    "depth": 6,
                    "l2_leaf_reg": 5.0,
                },
                {
                    "iterations": 340,
                    "learning_rate": 0.03,
                    "depth": 7,
                    "l2_leaf_reg": 6.0,
                },
            ],
        }

    return {
        "lgbm": [
            {
                "n_estimators": 420,
                "learning_rate": 0.03,
                "num_leaves": 63,
                "subsample": 0.90,
                "colsample_bytree": 0.90,
                "min_child_samples": 25,
            },
            {
                "n_estimators": 520,
                "learning_rate": 0.025,
                "num_leaves": 95,
                "subsample": 0.85,
                "colsample_bytree": 0.90,
                "min_child_samples": 20,
            },
        ],
        "xgb": [
            {
                "n_estimators": 420,
                "learning_rate": 0.03,
                "max_depth": 6,
                "min_child_weight": 3,
                "subsample": 0.90,
                "colsample_bytree": 0.90,
                "reg_lambda": 1.0,
            },
            {
                "n_estimators": 520,
                "learning_rate": 0.025,
                "max_depth": 5,
                "min_child_weight": 2,
                "subsample": 0.90,
                "colsample_bytree": 0.90,
                "reg_lambda": 1.5,
            },
        ],
        "cat": [
            {
                "iterations": 420,
                "learning_rate": 0.03,
                "depth": 6,
                "l2_leaf_reg": 5.0,
            },
            {
                "iterations": 520,
                "learning_rate": 0.025,
                "depth": 7,
                "l2_leaf_reg": 6.0,
            },
        ],
    }


def build_regressor(model_name, params, seed=42):
    if model_name == "lgbm":
        return LGBMRegressor(
            objective="regression",
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
            **params,
        )

    if model_name == "xgb":
        return XGBRegressor(
            objective="reg:squarederror",
            random_state=seed,
            n_jobs=-1,
            tree_method="hist",
            **params,
        )

    if model_name == "cat":
        return CatBoostRegressor(
            loss_function="RMSE",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            **params,
        )

    raise ValueError(f"Unknown model_name: {model_name}")


def simple_symmetry_round_decode(match_df, pred_team_cont, pred_opp_cont):
    pred_team_cont = np.asarray(pred_team_cont, dtype=float)
    pred_opp_cont = np.asarray(pred_opp_cont, dtype=float)

    pred_team_int = np.zeros(len(match_df), dtype=int)
    pred_opp_int = np.zeros(len(match_df), dtype=int)

    work = match_df[["match_id", "Id"]].copy()
    work["pred_team"] = pred_team_cont
    work["pred_opp"] = pred_opp_cont

    for _, g in work.groupby("match_id", sort=False):
        g = g.sort_values("Id")
        idx = g.index.to_list()

        if len(idx) != 2:
            raise ValueError("simple_symmetry_round_decode mengharapkan tepat 2 row per match.")

        i, j = idx
        lam_a = np.nanmean([pred_team_cont[i], pred_opp_cont[j]])
        lam_b = np.nanmean([pred_opp_cont[i], pred_team_cont[j]])

        ga = int(np.clip(np.round(lam_a), 0, 15))
        gb = int(np.clip(np.round(lam_b), 0, 15))

        pred_team_int[i] = ga
        pred_opp_int[i] = gb
        pred_team_int[j] = gb
        pred_opp_int[j] = ga

    return pred_team_int, pred_opp_int


def tune_one_model_family_for_gender(
    train_gender_df,
    folds_gender,
    feature_cols,
    categorical_cols,
    model_name,
    candidate_list,
    seed=42,
):
    fold0 = folds_gender[0]
    train_part = train_gender_df.loc[fold0["train_idx"]].reset_index(drop=True)
    valid_part = train_gender_df.loc[fold0["valid_idx"]].reset_index(drop=True)

    X_train, prep = fit_preprocessor(train_part, feature_cols, categorical_cols)
    X_valid = transform_with_preprocessor(valid_part, prep)

    y_train_team = train_part["team_goals"].to_numpy()
    y_train_opp = train_part["opp_goals"].to_numpy()

    y_valid_team = valid_part["team_goals"].to_numpy()
    y_valid_opp = valid_part["opp_goals"].to_numpy()

    best_score = np.inf
    best_params = None
    history_rows = []

    for candidate_id, params in enumerate(candidate_list):
        model_team = build_regressor(model_name, params, seed=seed)
        model_opp = build_regressor(model_name, params, seed=seed)

        model_team.fit(X_train, y_train_team)
        model_opp.fit(X_train, y_train_opp)

        pred_team_cont = np.clip(model_team.predict(X_valid), 0.0, None)
        pred_opp_cont = np.clip(model_opp.predict(X_valid), 0.0, None)

        pred_team_int, pred_opp_int = simple_symmetry_round_decode(
            valid_part,
            pred_team_cont,
            pred_opp_cont,
        )

        score = awmae_score(
            y_valid_team,
            y_valid_opp,
            pred_team_int,
            pred_opp_int,
            valid_part["tournament"].to_numpy(),
        )

        history_rows.append(
            {
                "model_name": model_name,
                "candidate_id": candidate_id,
                "awmae_fold0": score,
                "params": params,
            }
        )

        if score < best_score:
            best_score = score
            best_params = copy.deepcopy(params)

    history_df = pd.DataFrame(history_rows).sort_values("awmae_fold0").reset_index(drop=True)
    return best_params, history_df


def tune_all_models_for_gender(train_gender_df, folds_gender, feature_cols, categorical_cols, fast_mode=True, seed=42):
    candidates = get_model_candidates(fast_mode=fast_mode)
    best_params = {}
    tuning_tables = {}

    for model_name, candidate_list in candidates.items():
        bp, hist = tune_one_model_family_for_gender(
            train_gender_df=train_gender_df,
            folds_gender=folds_gender,
            feature_cols=feature_cols,
            categorical_cols=categorical_cols,
            model_name=model_name,
            candidate_list=candidate_list,
            seed=seed,
        )
        best_params[model_name] = bp
        tuning_tables[model_name] = hist

    return best_params, tuning_tables


## 10. Run tuning per gender

Tuning di sini tetap “jujur” karena:
- pakai fold validation time-based,
- scoring setelah decode integer,
- lalu dihitung dengan AW-MAE.

Jadi objective tuning tidak berhenti di MAE continuous biasa.


In [12]:
gender_results = {}
for gender_value in sorted(train_fe["gender"].dropna().unique()):
    train_g = train_fe.loc[train_fe["gender"] == gender_value].reset_index(drop=True).copy()
    test_g = test_fe.loc[test_fe["gender"] == gender_value].reset_index(drop=True).copy()

    folds_g = make_match_time_folds(
        train_g,
        n_splits=CONFIG["n_splits"],
        valid_frac=CONFIG["valid_frac"],
        min_train_frac=CONFIG["min_train_frac"],
    )

    print("=" * 80)
    print(f"GENDER = {gender_value}")
    print("train rows:", len(train_g), "| test rows:", len(test_g))
    print("fold summary:")
    display(fold_summary_df(folds_g))

    best_params_g, tuning_tables_g = tune_all_models_for_gender(
        train_gender_df=train_g,
        folds_gender=folds_g,
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        fast_mode=CONFIG["fast_mode"],
        seed=SEED,
    )

    print("best params:")
    print(json.dumps(best_params_g, indent=2))

    for model_name, table in tuning_tables_g.items():
        print(f"[{gender_value}] tuning table -> {model_name}")
        display(table)

    gender_results[gender_value] = {
        "train_df": train_g,
        "test_df": test_g,
        "folds": folds_g,
        "best_params": best_params_g,
        "tuning_tables": tuning_tables_g,
    }

print("Gender keys:", list(gender_results.keys()))


GENDER = M
train rows: 69966 | test rows: 28464
fold summary:


,fold,train_rows,valid_rows,train_matches,valid_matches,train_start,train_end,valid_start,valid_end
0,0,48978,6996,24489,3498,1872-11-30,2000-05-07,2000-05-07,2004-02-22
1,1,55974,6996,27987,3498,1872-11-30,2004-02-22,2004-02-22,2007-11-21
2,2,62970,6996,31485,3498,1872-11-30,2007-11-21,2007-11-21,2011-08-04


best params:
{
  "lgbm": {
    "n_estimators": 340,
    "learning_rate": 0.03,
    "num_leaves": 63,
    "subsample": 0.85,
    "colsample_bytree": 0.9,
    "min_child_samples": 30
  },
  "xgb": {
    "n_estimators": 260,
    "learning_rate": 0.04,
    "max_depth": 6,
    "min_child_weight": 4,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 1.0
  },
  "cat": {
    "iterations": 340,
    "learning_rate": 0.03,
    "depth": 7,
    "l2_leaf_reg": 6.0
  }
}
[M] tuning table -> lgbm


,model_name,candidate_id,awmae_fold0,params
0,lgbm,1,4.148048,"{'n_estimators': 340, 'learning_rate': 0.03, '..."
1,lgbm,0,4.201493,"{'n_estimators': 260, 'learning_rate': 0.04, '..."


[M] tuning table -> xgb


...

[W] tuning table -> xgb


,model_name,candidate_id,awmae_fold0,params
0,xgb,1,5.593503,"{'n_estimators': 340, 'learning_rate': 0.03, '..."
1,xgb,0,5.644360,"{'n_estimators': 260, 'learning_rate': 0.04, '..."


[W] tuning table -> cat


,model_name,candidate_id,awmae_fold0,params
0,cat,0,5.763926,"{'iterations': 260, 'learning_rate': 0.04, 'de..."
1,cat,1,5.814481,"{'iterations': 340, 'learning_rate': 0.03, 'de..."


Gender keys: ['M', 'W']


## 11. Cross-validated base modeling per gender

Bagian ini melatih:
- LightGBM,
- XGBoost,
- CatBoost,

secara **per gender** dan **per target** (`team_goals`, `opp_goals`), lalu menyimpan:
- OOF base predictions,
- fold-level validation predictions.

OOF inilah yang nanti dipakai untuk melatih Ridge stacker secara jujur.


In [13]:
BASE_MODEL_NAMES = ["lgbm", "xgb", "cat"]
STACK_FEATURE_COLUMNS = [
    "lgbm_team",
    "lgbm_opp",
    "xgb_team",
    "xgb_opp",
    "cat_team",
    "cat_opp",
]


def run_cv_base_models_for_gender(train_gender_df, folds_gender, feature_cols, categorical_cols, best_params, seed=42):
    train_gender_df = train_gender_df.reset_index(drop=True).copy()

    oof_df = pd.DataFrame(index=train_gender_df.index)
    covered_mask = np.zeros(len(train_gender_df), dtype=bool)

    for model_name in BASE_MODEL_NAMES:
        oof_team = np.full(len(train_gender_df), np.nan, dtype=float)
        oof_opp = np.full(len(train_gender_df), np.nan, dtype=float)

        for fold in folds_gender:
            tr_idx = fold["train_idx"]
            va_idx = fold["valid_idx"]
            covered_mask[va_idx] = True

            train_part = train_gender_df.loc[tr_idx].reset_index(drop=True)
            valid_part = train_gender_df.loc[va_idx].reset_index(drop=True)

            X_train, prep = fit_preprocessor(train_part, feature_cols, categorical_cols)
            X_valid = transform_with_preprocessor(valid_part, prep)

            y_train_team = train_part["team_goals"].to_numpy()
            y_train_opp = train_part["opp_goals"].to_numpy()

            model_team = build_regressor(model_name, best_params[model_name], seed=seed)
            model_opp = build_regressor(model_name, best_params[model_name], seed=seed)

            model_team.fit(X_train, y_train_team)
            model_opp.fit(X_train, y_train_opp)

            oof_team[va_idx] = np.clip(model_team.predict(X_valid), 0.0, None)
            oof_opp[va_idx] = np.clip(model_opp.predict(X_valid), 0.0, None)

        oof_df[f"{model_name}_team"] = oof_team
        oof_df[f"{model_name}_opp"] = oof_opp

    assert covered_mask.any(), "Tidak ada row validation yang tercakup."
    assert oof_df.loc[covered_mask].notna().all().all(), "Masih ada OOF kosong pada row yang seharusnya tercakup."

    return oof_df, covered_mask


def fit_full_models_for_gender(train_gender_df, feature_cols, categorical_cols, best_params, seed=42):
    train_gender_df = train_gender_df.reset_index(drop=True).copy()

    model_store = {}
    X_train, prep = fit_preprocessor(train_gender_df, feature_cols, categorical_cols)

    y_team = train_gender_df["team_goals"].to_numpy()
    y_opp = train_gender_df["opp_goals"].to_numpy()

    for model_name in BASE_MODEL_NAMES:
        model_team = build_regressor(model_name, best_params[model_name], seed=seed)
        model_opp = build_regressor(model_name, best_params[model_name], seed=seed)

        model_team.fit(X_train, y_team)
        model_opp.fit(X_train, y_opp)

        model_store[model_name] = {
            "preprocessor": prep,
            "team_model": model_team,
            "opp_model": model_opp,
        }

    return model_store


def predict_full_models_for_gender(test_gender_df, model_store):
    pred_df = pd.DataFrame(index=test_gender_df.index)

    for model_name in BASE_MODEL_NAMES:
        prep = model_store[model_name]["preprocessor"]
        X_test = transform_with_preprocessor(test_gender_df, prep)

        pred_df[f"{model_name}_team"] = np.clip(model_store[model_name]["team_model"].predict(X_test), 0.0, None)
        pred_df[f"{model_name}_opp"] = np.clip(model_store[model_name]["opp_model"].predict(X_test), 0.0, None)

    return pred_df


## 12. Ridge stacking

Stacker dilatih hanya memakai OOF base predictions.  
Ini penting supaya blending tidak “curang” karena melihat prediksi in-sample.


In [14]:
def fit_ridge_stackers(oof_base_df, y_team, y_opp, covered_mask):
    X_stack = oof_base_df.loc[covered_mask, STACK_FEATURE_COLUMNS].to_numpy()

    ridge_team = Ridge(alpha=1.0)
    ridge_opp = Ridge(alpha=1.0)

    ridge_team.fit(X_stack, y_team[covered_mask])
    ridge_opp.fit(X_stack, y_opp[covered_mask])

    return {
        "team_model": ridge_team,
        "opp_model": ridge_opp,
    }


def apply_ridge_stackers(base_pred_df, stacker_store):
    X_stack = base_pred_df[STACK_FEATURE_COLUMNS].to_numpy()

    pred_team = np.clip(stacker_store["team_model"].predict(X_stack), 0.0, None)
    pred_opp = np.clip(stacker_store["opp_model"].predict(X_stack), 0.0, None)

    return pred_team, pred_opp


## 13. Dixon-Coles rho + MBR decoding

Setelah continuous prediction didapat, kita:
1. estimasi rho Dixon-Coles per gender dari OOF,
2. lakukan decode pada level match,
3. pastikan dua row dalam satu match selalu konsisten.

Di sini decode dilakukan dengan prinsip MBR sederhana:
- bangun distribusi skor joint berbasis Poisson + DC adjustment,
- pilih skor integer dengan expected loss minimum menurut loss function yang selaras dengan evaluator offline.


In [15]:
MAX_DECODE_GOALS = int(CONFIG["max_decode_goals"])


def poisson_pmf(k, lam):
    lam = max(float(lam), 1e-8)
    return math.exp(-lam) * (lam ** int(k)) / math.factorial(int(k))


def dixon_coles_tau(x, y, lam_home, lam_away, rho):
    if x == 0 and y == 0:
        return max(1e-12, 1.0 - lam_home * lam_away * rho)
    if x == 0 and y == 1:
        return max(1e-12, 1.0 + lam_home * rho)
    if x == 1 and y == 0:
        return max(1e-12, 1.0 + lam_away * rho)
    if x == 1 and y == 1:
        return max(1e-12, 1.0 - rho)
    return 1.0


def build_joint_dc_matrix(lam_home, lam_away, rho, max_goals=8):
    grid = np.zeros((max_goals + 1, max_goals + 1), dtype=float)

    for x in range(max_goals + 1):
        for y in range(max_goals + 1):
            base = poisson_pmf(x, lam_home) * poisson_pmf(y, lam_away)
            tau = dixon_coles_tau(x, y, lam_home, lam_away, rho)
            grid[x, y] = base * tau

    total = grid.sum()
    if total <= 0:
        grid[:] = 1.0 / grid.size
    else:
        grid /= total
    return grid


LOSS_TENSOR = np.zeros(
    (
        MAX_DECODE_GOALS + 1,
        MAX_DECODE_GOALS + 1,
        MAX_DECODE_GOALS + 1,
        MAX_DECODE_GOALS + 1,
    ),
    dtype=float,
)

for yt in range(MAX_DECODE_GOALS + 1):
    for yo in range(MAX_DECODE_GOALS + 1):
        for pt in range(MAX_DECODE_GOALS + 1):
            for po in range(MAX_DECODE_GOALS + 1):
                LOSS_TENSOR[yt, yo, pt, po] = official_match_loss(
                    yt,
                    yo,
                    pt,
                    po,
                    tournament_weight=1.0,
                )


def mbr_decode_one_match(lam_team, lam_opp, rho, max_goals=8):
    lam_team = max(float(lam_team), 1e-6)
    lam_opp = max(float(lam_opp), 1e-6)

    joint = build_joint_dc_matrix(lam_team, lam_opp, rho, max_goals=max_goals)
    expected_loss = np.tensordot(joint, LOSS_TENSOR, axes=([0, 1], [0, 1]))
    best_idx = np.unravel_index(np.argmin(expected_loss), expected_loss.shape)
    return int(best_idx[0]), int(best_idx[1])


def build_match_level_lambda_frame(meta_df, pred_team_cont, pred_opp_cont):
    meta_df = meta_df.reset_index(drop=True).copy()
    pred_team_cont = np.asarray(pred_team_cont, dtype=float)
    pred_opp_cont = np.asarray(pred_opp_cont, dtype=float)

    rows = []
    for match_id, g in meta_df.groupby("match_id", sort=False):
        g = g.sort_values("Id")
        idx = g.index.to_list()

        if len(idx) != 2:
            raise ValueError("Setiap match harus 2 row saat build_match_level_lambda_frame.")

        i, j = idx
        lam_a = float(np.nanmean([pred_team_cont[i], pred_opp_cont[j]]))
        lam_b = float(np.nanmean([pred_opp_cont[i], pred_team_cont[j]]))

        rows.append(
            {
                "match_id": match_id,
                "row_i": i,
                "row_j": j,
                "lam_a": max(lam_a, 1e-6),
                "lam_b": max(lam_b, 1e-6),
                "true_a": int(meta_df.loc[i, "team_goals"]) if "team_goals" in meta_df.columns else np.nan,
                "true_b": int(meta_df.loc[i, "opp_goals"]) if "opp_goals" in meta_df.columns else np.nan,
            }
        )

    return pd.DataFrame(rows)


def estimate_dixon_coles_rho(meta_df, pred_team_cont, pred_opp_cont, rho_grid=None):
    if rho_grid is None:
        rho_grid = np.linspace(-0.15, 0.15, 31)

    match_level = build_match_level_lambda_frame(meta_df, pred_team_cont, pred_opp_cont)

    best_rho = 0.0
    best_nll = np.inf

    for rho in rho_grid:
        nlls = []
        for _, row in match_level.iterrows():
            x = int(row["true_a"])
            y = int(row["true_b"])
            lam_a = max(float(row["lam_a"]), 1e-6)
            lam_b = max(float(row["lam_b"]), 1e-6)

            prob = poisson_pmf(x, lam_a) * poisson_pmf(y, lam_b) * dixon_coles_tau(x, y, lam_a, lam_b, rho)
            prob = max(prob, 1e-12)
            nlls.append(-math.log(prob))

        mean_nll = float(np.mean(nlls))
        if mean_nll < best_nll:
            best_nll = mean_nll
            best_rho = float(rho)

    return best_rho, best_nll


def apply_match_level_mbr_decode(meta_df, pred_team_cont, pred_opp_cont, rho, max_goals=8):
    meta_df = meta_df.reset_index(drop=True).copy()
    pred_team_cont = np.asarray(pred_team_cont, dtype=float)
    pred_opp_cont = np.asarray(pred_opp_cont, dtype=float)

    pred_team_int = np.zeros(len(meta_df), dtype=int)
    pred_opp_int = np.zeros(len(meta_df), dtype=int)

    for match_id, g in meta_df.groupby("match_id", sort=False):
        g = g.sort_values("Id")
        idx = g.index.to_list()

        if len(idx) != 2:
            raise ValueError("Setiap match harus 2 row saat MBR decode.")

        i, j = idx
        lam_a = float(np.nanmean([pred_team_cont[i], pred_opp_cont[j]]))
        lam_b = float(np.nanmean([pred_opp_cont[i], pred_team_cont[j]]))

        ga, gb = mbr_decode_one_match(lam_a, lam_b, rho, max_goals=max_goals)

        pred_team_int[i] = ga
        pred_opp_int[i] = gb
        pred_team_int[j] = gb
        pred_opp_int[j] = ga

    pred_team_int = np.clip(pred_team_int, 0, None)
    pred_opp_int = np.clip(pred_opp_int, 0, None)
    return pred_team_int.astype(int), pred_opp_int.astype(int)


## 14. Full training per gender + stacking + decode

Urutannya:
1. CV base models -> OOF,
2. Ridge stacker dilatih di atas OOF,
3. full-data refit untuk prediksi test freeze-safe,
4. estimasi rho per gender dari OOF,
5. decode OOF dan test dengan MBR.


In [17]:
global_oof_covered = np.zeros(len(train_fe), dtype=bool)

global_oof_team = np.full(len(train_fe), np.nan, dtype=float)
global_oof_opp = np.full(len(train_fe), np.nan, dtype=float)
global_oof_covered = np.zeros(len(train_fe), dtype=bool)

global_test_team = np.full(len(test_fe), np.nan, dtype=float)
global_test_opp = np.full(len(test_fe), np.nan, dtype=float)

validation_rows = []
gender_breakdown_rows = []
tournament_breakdown_rows = []

for gender_value, info in gender_results.items():
    print("=" * 100)
    print("PROCESSING GENDER:", gender_value)

    train_g = info["train_df"].copy()
    test_g = info["test_df"].copy()
    folds_g = info["folds"]
    best_params_g = info["best_params"]

    # 1) OOF base predictions via CV
    oof_base_g, covered_mask_g = run_cv_base_models_for_gender(
        train_gender_df=train_g,
        folds_gender=folds_g,
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        best_params=best_params_g,
        seed=SEED,
    )

    stacker_g = fit_ridge_stackers(
        oof_base_df=oof_base_g,
        y_team=train_g["team_goals"].to_numpy(),
        y_opp=train_g["opp_goals"].to_numpy(),
        covered_mask=covered_mask_g,
    )

    oof_stack_team_cont = np.full(len(train_g), np.nan, dtype=float)
    oof_stack_opp_cont = np.full(len(train_g), np.nan, dtype=float)

    tmp_team, tmp_opp = apply_ridge_stackers(oof_base_g.loc[covered_mask_g].copy(), stacker_g)
    oof_stack_team_cont[covered_mask_g] = tmp_team
    oof_stack_opp_cont[covered_mask_g] = tmp_opp
    # 3) Estimate rho from OOF
    rho_g, rho_nll_g = estimate_dixon_coles_rho(
        meta_df=train_g.loc[covered_mask_g].reset_index(drop=True),
        pred_team_cont=oof_stack_team_cont[covered_mask_g],
        pred_opp_cont=oof_stack_opp_cont[covered_mask_g],
        rho_grid=np.linspace(-0.15, 0.15, 31),
    )

    oof_team_int_cov, oof_opp_int_cov = apply_match_level_mbr_decode(
        meta_df=train_g.loc[covered_mask_g].reset_index(drop=True),
        pred_team_cont=oof_stack_team_cont[covered_mask_g],
        pred_opp_cont=oof_stack_opp_cont[covered_mask_g],
        rho=rho_g,
        max_goals=MAX_DECODE_GOALS,
    )

    oof_team_int = np.full(len(train_g), np.nan)
    oof_opp_int = np.full(len(train_g), np.nan)
    oof_team_int[covered_mask_g] = oof_team_int_cov
    oof_opp_int[covered_mask_g] = oof_opp_int_cov

    # 5) Refit full-data base models and predict freeze-safe test
    full_model_store_g = fit_full_models_for_gender(
        train_gender_df=train_g,
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        best_params=best_params_g,
        seed=SEED,
    )
    test_base_g = predict_full_models_for_gender(
        test_gender_df=test_g,
        model_store=full_model_store_g,
    )

    test_stack_team_cont, test_stack_opp_cont = apply_ridge_stackers(test_base_g, stacker_g)

    test_team_int, test_opp_int = apply_match_level_mbr_decode(
        meta_df=test_g,
        pred_team_cont=test_stack_team_cont,
        pred_opp_cont=test_stack_opp_cont,
        rho=rho_g,
        max_goals=MAX_DECODE_GOALS,
    )

    # Fold-level validation
    for fold in folds_g:
        va_idx = fold["valid_idx"]
        score = awmae_score(
            train_g.loc[va_idx, "team_goals"].to_numpy(),
            train_g.loc[va_idx, "opp_goals"].to_numpy(),
            oof_team_int[va_idx],
            oof_opp_int[va_idx],
            train_g.loc[va_idx, "tournament"].to_numpy(),
        )
        validation_rows.append(
            {
                "gender": gender_value,
                "fold": fold["fold"],
                "awmae": score,
                "n_valid_rows": len(va_idx),
                "valid_start": fold["valid_start"],
                "valid_end": fold["valid_end"],
            }
        )

    # Gender breakdown
    eval_mask_g = covered_mask_g

    gender_breakdown_rows.append(
        {
            "gender": gender_value,
            "n_eval_rows": int(eval_mask_g.sum()),
            "coverage_ratio": float(eval_mask_g.mean()),
            "awmae": awmae_score(
                train_g.loc[eval_mask_g, "team_goals"].to_numpy(),
                train_g.loc[eval_mask_g, "opp_goals"].to_numpy(),
                oof_team_int[eval_mask_g].astype(int),
                oof_opp_int[eval_mask_g].astype(int),
                train_g.loc[eval_mask_g, "tournament"].to_numpy(),
            ),
            "exact_hit_rate": exact_hit_rate(
                train_g.loc[eval_mask_g, "team_goals"].to_numpy(),
                train_g.loc[eval_mask_g, "opp_goals"].to_numpy(),
                oof_team_int[eval_mask_g].astype(int),
                oof_opp_int[eval_mask_g].astype(int),
            ),
            "outcome_hit_rate": outcome_hit_rate(
                train_g.loc[eval_mask_g, "team_goals"].to_numpy(),
                train_g.loc[eval_mask_g, "opp_goals"].to_numpy(),
                oof_team_int[eval_mask_g].astype(int),
                oof_opp_int[eval_mask_g].astype(int),
            ),
            "gd_hit_rate": gd_hit_rate(
                train_g.loc[eval_mask_g, "team_goals"].to_numpy(),
                train_g.loc[eval_mask_g, "opp_goals"].to_numpy(),
                oof_team_int[eval_mask_g].astype(int),
                oof_opp_int[eval_mask_g].astype(int),
            ),
            "rho": rho_g,
        }
    )

    # Tournament weight breakdown
    temp_eval = train_g.loc[covered_mask_g, ["tournament", "team_goals", "opp_goals"]].copy()
    temp_eval["pred_team_goals"] = oof_team_int[covered_mask_g].astype(int)
    temp_eval["pred_opp_goals"] = oof_opp_int[covered_mask_g].astype(int)
    temp_eval["tournament_weight"] = temp_eval["tournament"].map(get_tournament_weight)
    for weight_value, wdf in temp_eval.groupby("tournament_weight", dropna=False):
        tournament_breakdown_rows.append(
            {
                "gender": gender_value,
                "tournament_weight": weight_value,
                "awmae": awmae_score(
                    wdf["team_goals"].to_numpy(),
                    wdf["opp_goals"].to_numpy(),
                    wdf["pred_team_goals"].to_numpy(),
                    wdf["pred_opp_goals"].to_numpy(),
                    wdf["tournament"].to_numpy(),
                ),
                "exact_hit_rate": exact_hit_rate(
                    wdf["team_goals"].to_numpy(),
                    wdf["opp_goals"].to_numpy(),
                    wdf["pred_team_goals"].to_numpy(),
                    wdf["pred_opp_goals"].to_numpy(),
                ),
                "outcome_hit_rate": outcome_hit_rate(
                    wdf["team_goals"].to_numpy(),
                    wdf["opp_goals"].to_numpy(),
                    wdf["pred_team_goals"].to_numpy(),
                    wdf["pred_opp_goals"].to_numpy(),
                ),
                "gd_hit_rate": gd_hit_rate(
                    wdf["team_goals"].to_numpy(),
                    wdf["opp_goals"].to_numpy(),
                    wdf["pred_team_goals"].to_numpy(),
                    wdf["pred_opp_goals"].to_numpy(),
                ),
                "n_rows": len(wdf),
            }
        )

    # Write back into global arrays
    train_global_idx = train_g["row_idx_global"].to_numpy()
    test_global_idx = test_g["row_idx_global"].to_numpy()

    global_oof_team[train_global_idx] = oof_team_int
    global_oof_opp[train_global_idx] = oof_opp_int
    global_oof_covered[train_global_idx] = covered_mask_g

    global_test_team[test_global_idx] = test_team_int
    global_test_opp[test_global_idx] = test_opp_int

    # Store everything useful
    info["oof_base"] = oof_base_g
    info["stacker_store"] = stacker_g
    info["rho"] = rho_g
    info["full_model_store"] = full_model_store_g
    info["test_base"] = test_base_g
    info["oof_stack_team_cont"] = oof_stack_team_cont
    info["oof_stack_opp_cont"] = oof_stack_opp_cont
    info["oof_team_int"] = oof_team_int
    info["oof_opp_int"] = oof_opp_int
    info["test_stack_team_cont"] = test_stack_team_cont
    info["test_stack_opp_cont"] = test_stack_opp_cont
    info["test_team_int"] = test_team_int
    info["test_opp_int"] = test_opp_int

    gc.collect()

assert global_oof_covered.any(), "Tidak ada row OOF yang covered."
assert np.isfinite(global_oof_team[global_oof_covered]).all()
assert np.isfinite(global_oof_opp[global_oof_covered]).all()

assert np.isfinite(global_test_team).all()
assert np.isfinite(global_test_opp).all()


PROCESSING GENDER: M
PROCESSING GENDER: W


## 15. Validation analysis

Karena kita tidak boleh melihat ground truth test, maka semua analisis kualitas model difokuskan penuh pada validation/OOF dari train.


In [19]:
validation_df = pd.DataFrame(validation_rows).sort_values(["gender", "fold"]).reset_index(drop=True)
gender_breakdown_df = pd.DataFrame(gender_breakdown_rows).sort_values("gender").reset_index(drop=True)
tournament_breakdown_df = (
    pd.DataFrame(tournament_breakdown_rows)
    .sort_values(["gender", "tournament_weight"])
    .reset_index(drop=True)
)

print("=== AW-MAE PER FOLD ===")
display(validation_df)

print("=== MEAN / STD AW-MAE ===")
summary_stats = validation_df.groupby("gender")["awmae"].agg(["mean", "std", "count"]).reset_index()
display(summary_stats)

print("=== BREAKDOWN PER GENDER ===")
display(gender_breakdown_df)

print("=== BREAKDOWN PER TOURNAMENT WEIGHT ===")
display(tournament_breakdown_df)

overall_awmae = awmae_score(
    train_fe.loc[global_oof_covered, "team_goals"].to_numpy(),
    train_fe.loc[global_oof_covered, "opp_goals"].to_numpy(),
    global_oof_team[global_oof_covered].astype(int),
    global_oof_opp[global_oof_covered].astype(int),
    train_fe.loc[global_oof_covered, "tournament"].to_numpy(),
)

overall_exact = exact_hit_rate(
    train_fe["team_goals"].to_numpy(),
    train_fe["opp_goals"].to_numpy(),
    global_oof_team.astype(int),
    global_oof_opp.astype(int),
)
overall_outcome = outcome_hit_rate(
    train_fe["team_goals"].to_numpy(),
    train_fe["opp_goals"].to_numpy(),
    global_oof_team.astype(int),
    global_oof_opp.astype(int),
)
overall_gd = gd_hit_rate(
    train_fe["team_goals"].to_numpy(),
    train_fe["opp_goals"].to_numpy(),
    global_oof_team.astype(int),
    global_oof_opp.astype(int),
)

print("=== OVERALL OOF SUMMARY ===")
print(f"AW-MAE          : {overall_awmae:.6f}")
print(f"Exact-hit rate  : {overall_exact:.4f}")
print(f"Outcome-hit rate: {overall_outcome:.4f}")
print(f"GD-hit rate     : {overall_gd:.4f}")


=== AW-MAE PER FOLD ===


,gender,fold,awmae,n_valid_rows,valid_start,valid_end
0,M,0,4.092797,6996,2000-05-07,2004-02-22
1,M,1,3.876774,6996,2004-02-22,2007-11-21
2,M,2,3.741339,6996,2007-11-21,2011-08-04
3,W,0,5.183729,880,2007-04-07,2008-06-26
4,W,1,5.344235,880,2008-06-26,2010-05-12
5,W,2,4.944373,880,2010-05-13,2011-07-17


=== MEAN / STD AW-MAE ===


,gender,mean,std,count
0,M,3.903637,0.177262,3
1,W,5.157446,0.201222,3


=== BREAKDOWN PER GENDER ===


,gender,n_eval_rows,coverage_ratio,awmae,exact_hit_rate,outcome_hit_rate,gd_hit_rate,rho
0,M,20988,0.299974,3.903637,0.116733,0.567467,0.251763,-0.06
1,W,2640,0.299796,5.157446,0.105303,0.703030,0.188636,0.11


=== BREAKDOWN PER TOURNAMENT WEIGHT ===


,gender,tournament_weight,awmae,exact_hit_rate,outcome_hit_rate,gd_hit_rate,n_rows
0,M,0.96,2.609720,0.122225,0.513711,0.272656,7658
1,M,1.20,3.593752,0.104512,0.575671,0.230725,3502
2,M,1.80,4.812390,0.112574,0.595157,0.236194,4708
3,M,2.00,5.215282,0.120703,0.616797,0.249219,5120
4,W,0.96,2.262560,0.035714,0.892857,0.107143,56
5,W,1.20,3.997479,0.116700,0.659960,0.199195,994
6,W,1.80,6.149967,0.102752,0.704587,0.172477,1090
7,W,2.00,5.623990,0.096000,0.764000,0.212000,500


=== OVERALL OOF SUMMARY ===
AW-MAE          : 4.043727
Exact-hit rate  : 0.0346
Outcome-hit rate: 0.3233
GD-hit rate     : 0.2220


### Insight singkat

Kalau hasil OOF masih belum cukup rendah, biasanya langkah berikut yang paling layak dicoba setelah notebook ini adalah:
1. memperkaya state/history tanpa mengorbankan submission-safety,
2. membuat tuning per-gender yang lebih dalam,
3. menambah post-processing berbasis struktur pertandingan/tournament group,
4. membandingkan freeze-safe baseline vs recursive-safe research branch.

Yang penting, setelah notebook ini, pondasinya sudah bersih dulu.


## 16. Generate submission

Output akhir wajib:
- tidak ada NaN,
- tidak ada nilai negatif,
- jumlah row sama dengan test,
- urutan `Id` konsisten dengan sample submission.


In [20]:
submission = pd.DataFrame(
    {
        "Id": test_fe["Id"].values,
        "team_goals": global_test_team.astype(int),
        "opp_goals": global_test_opp.astype(int),
    }
)

submission["team_goals"] = submission["team_goals"].clip(lower=0).astype(int)
submission["opp_goals"] = submission["opp_goals"].clip(lower=0).astype(int)

submission = sample_submission[["Id"]].merge(
    submission,
    on="Id",
    how="left",
    validate="one_to_one",
)

assert len(submission) == len(test_fe), "Jumlah row submission tidak sama dengan test."
assert submission["Id"].equals(sample_submission["Id"]), "Urutan Id submission tidak sama dengan sample submission."
assert submission[["team_goals", "opp_goals"]].notna().all().all(), "Masih ada NaN di submission."
assert (submission[["team_goals", "opp_goals"]] >= 0).all().all(), "Masih ada prediksi negatif."

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print("submission saved to:", submission_path)
display(submission.head())


submission saved to: submission.csv


,Id,team_goals,opp_goals
0,M034984_Seychelles,2,1
1,M034984_Mauritius,1,2
2,M034985_Comoros,1,2
3,M034985_Maldives,2,1
4,M034986_Réunion,2,1


## 17. Optional branch — recursive_safe_inference (research-only)

Branch ini **opsional** dan **bukan baseline utama**.  
Tujuannya adalah menguji skenario research di mana test diproses berurutan berdasarkan tanggal, lalu state di-update memakai prediksi test sebelumnya.

Baseline resmi notebook ini tetap:
- **freeze-safe**
- hanya memakai last-known state dari train
- tidak meng-update state dengan prediksi test saat membuat submission utama.


In [21]:
def update_states_from_predicted_match(match_rows_df, pred_team_int, pred_opp_int, team_states, h2h_states):
    match_rows_df = match_rows_df.sort_values("Id").reset_index(drop=True).copy()
    if len(match_rows_df) != 2:
        raise ValueError("Setiap match harus 2 row untuk recursive update.")

    current_date = pd.Timestamp(match_rows_df.loc[0, "date"])

    # write temporary score columns to reuse update logic
    work = match_rows_df.copy()
    work["team_goals"] = pred_team_int
    work["opp_goals"] = pred_opp_int

    for _, row in work.iterrows():
        team = row["team"]
        opp = row["opponent"]
        goals_for = int(row["team_goals"])
        goals_against = int(row["opp_goals"])

        state = team_states[team]
        state["points"].append(points_for_score(goals_for, goals_against))
        state["gd"].append(goals_for - goals_against)
        state["gf"].append(goals_for)
        state["ga"].append(goals_against)
        state["win"].append(1 if goals_for > goals_against else 0)
        state["last_date"] = current_date
        state["matches_played"] += 1.0

        h2h = h2h_states[(team, opp)]
        h2h["points"].append(points_for_score(goals_for, goals_against))
        h2h["gd"].append(goals_for - goals_against)
        h2h["gf"].append(goals_for)
        h2h["ga"].append(goals_against)
        h2h["matches_played"] += 1.0

    row0 = work.iloc[0]
    row1 = work.iloc[1]
    team_a = row0["team"]
    team_b = row1["team"]

    elo_a_new, elo_b_new = elo_update(
        team_states[team_a]["elo"],
        team_states[team_b]["elo"],
        int(row0["team_goals"]),
        int(row0["opp_goals"]),
        is_home=int(row0["is_home"]),
    )
    team_states[team_a]["elo"] = elo_a_new
    team_states[team_b]["elo"] = elo_b_new

    pi_a_new, pi_b_new = pi_update(
        team_states[team_a]["pi"],
        team_states[team_b]["pi"],
        int(row0["team_goals"]),
        int(row0["opp_goals"]),
    )
    team_states[team_a]["pi"] = pi_a_new
    team_states[team_b]["pi"] = pi_b_new


def recursive_safe_inference_for_gender(
    test_gender_df,
    full_model_store,
    stacker_store,
    rho,
    base_team_states,
    base_h2h_states,
    feature_cols,
    categorical_cols,
):
    mutable_team_states = copy.deepcopy(base_team_states)
    mutable_h2h_states = copy.deepcopy(base_h2h_states)

    pred_team_all = np.zeros(len(test_gender_df), dtype=int)
    pred_opp_all = np.zeros(len(test_gender_df), dtype=int)

    for _, match_rows in test_gender_df.groupby("match_id", sort=False):
        match_rows = match_rows.sort_values(["date", "match_id", "Id"]).reset_index(drop=False)
        local_df = match_rows.copy()

        # features for current match from current mutable state
        live_hist = make_test_features_freeze_safe(local_df.drop(columns=["index"]), mutable_team_states, mutable_h2h_states)
        live_fe = add_submission_safe_context_features(live_hist)

        base_pred_live = pd.DataFrame(index=live_fe.index)
        for model_name in BASE_MODEL_NAMES:
            prep = full_model_store[model_name]["preprocessor"]
            X_live = transform_with_preprocessor(live_fe, prep)

            base_pred_live[f"{model_name}_team"] = np.clip(
                full_model_store[model_name]["team_model"].predict(X_live), 0.0, None
            )
            base_pred_live[f"{model_name}_opp"] = np.clip(
                full_model_store[model_name]["opp_model"].predict(X_live), 0.0, None
            )

        live_team_cont, live_opp_cont = apply_ridge_stackers(base_pred_live, stacker_store)

        live_team_int, live_opp_int = apply_match_level_mbr_decode(
            meta_df=live_fe.reset_index(drop=True),
            pred_team_cont=live_team_cont,
            pred_opp_cont=live_opp_cont,
            rho=rho,
            max_goals=MAX_DECODE_GOALS,
        )

        original_indices = match_rows["index"].to_numpy()
        pred_team_all[original_indices] = live_team_int
        pred_opp_all[original_indices] = live_opp_int

        update_states_from_predicted_match(
            match_rows_df=live_fe,
            pred_team_int=live_team_int,
            pred_opp_int=live_opp_int,
            team_states=mutable_team_states,
            h2h_states=mutable_h2h_states,
        )

    return pred_team_all, pred_opp_all


if CONFIG["run_optional_recursive_branch"]:
    recursive_predictions = {}
    for gender_value, info in gender_results.items():
        test_g = info["test_df"].copy().reset_index(drop=True)
        base_team_states = train_team_states_final
        base_h2h_states = train_h2h_states_final

        pred_team_rec, pred_opp_rec = recursive_safe_inference_for_gender(
            test_gender_df=test_g,
            full_model_store=info["full_model_store"],
            stacker_store=info["stacker_store"],
            rho=info["rho"],
            base_team_states=base_team_states,
            base_h2h_states=base_h2h_states,
            feature_cols=feature_cols,
            categorical_cols=categorical_cols,
        )

        recursive_predictions[gender_value] = {
            "team_goals": pred_team_rec,
            "opp_goals": pred_opp_rec,
        }

    print("Recursive branch selesai dijalankan.")
else:
    print("Recursive branch dinonaktifkan. Baseline utama tetap freeze-safe.")


Recursive branch dinonaktifkan. Baseline utama tetap freeze-safe.


## 18. Experiment conclusion

Eksperimen ini menghasilkan fondasi V5 yang lebih bersih dan jujur, dengan poin utama:
- seluruh feature engineering test bebas dari ground truth test,
- history dibangun per match agar tidak bocor antar 2 row dalam pertandingan yang sama,
- validation dilakukan time-based dan match-level,
- model tetap mempertahankan struktur kuat:
  - per-gender,
  - 3 booster,
  - ridge stacking,
  - Dixon-Coles rho,
  - MBR decoding.

Dengan notebook ini, langkah eksperimen berikutnya bisa dilakukan di atas fondasi yang sudah submission-safe, bukan di atas offline score yang terlalu optimistis akibat leakage.
